# Results Summary

This notebook aggregates experiment outputs and produces compact visual summaries for interpretation.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
results_csv = repo_root / "results" / "notebook_experiments.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
else:
    print("No experiment CSV found; generating synthetic placeholder data.")
    rng = np.random.default_rng(42)
    df = pd.DataFrame(
        {
            "seed": np.tile(np.arange(10), 2),
            "condition": ["baseline"] * 10 + ["distinction"] * 10,
            "divergence": np.concatenate([rng.normal(4.5, 0.5, 10), rng.normal(3.7, 0.4, 10)]),
            "variability": np.concatenate([rng.normal(0.08, 0.01, 10), rng.normal(0.06, 0.01, 10)]),
            "compliance": np.concatenate([rng.normal(0.45, 0.05, 10), rng.normal(0.68, 0.06, 10)]),
        }
    )

df.head()


In [ ]:
summary = (
    df.groupby("condition")
    .agg(
        divergence_mean=("divergence", "mean"),
        divergence_std=("divergence", "std"),
        variability_mean=("variability", "mean"),
        compliance_mean=("compliance", "mean"),
    )
    .round(4)
)

summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ["divergence", "variability", "compliance"]
for i, metric in enumerate(metrics):
    ax = axes[i]
    subset = [df.loc[df["condition"] == cond, metric] for cond in ["baseline", "distinction"]]
    ax.boxplot(subset, labels=["baseline", "distinction"])
    ax.set_title(metric.capitalize())
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()


## Interpretation Checklist

- Compare whether the distinction condition improves compliance.
- Check if divergence and variability move in expected directions.
- Use confidence intervals or statistical tests for stronger conclusions in formal reports.